# ESM-IF / ProteinDPO Tool Example

This notebook demonstrates how to use **ESM-IF1** and **ProteinDPO** for structure-conditioned protein sequence design (inverse folding) and scoring.

## What is ESM-IF?

[ESM-IF1](https://doi.org/10.1101/2022.04.10.487779) (Inverse Folding) is a 142M-parameter model that designs protein sequences conditioned on backbone structure using a GVP-GNN encoder and Transformer decoder.

## What is ProteinDPO?

[ProteinDPO](https://doi.org/10.1101/2024.05.20.595026) is a fine-tuned version of ESM-IF1 aligned to experimental fitness data via Direct Preference Optimization. It is optimized for designing stable protein sequences.

### Key Features:
- **Structure-conditioned inverse folding** -- Design sequences compatible with a target backbone
- **Multi-chain complex support** -- Scoring and sampling with full complex structural context
- **Switchable weights** -- Use vanilla ESM-IF1 or stability-optimized ProteinDPO weights
- **Sequence scoring** -- Evaluate how well a sequence fits a given structure

## Imports

In [ ]:
from bio_programming_tools import (
    InverseFoldingInput,
    InverseFoldingStructureInput,
    ESMIF1SampleConfig,
    ESMIF1ScoringInput,
    ESMIF1ScoringConfig,
    SequenceStructurePair,
    run_esm_if1_sample,
    run_esm_if1_score,
)
from bio_programming_tools.entities.structures.examples import get_gfp_structure

## 1. Sample Sequences with ProteinDPO

Generate sequences conditioned on a backbone structure using ProteinDPO (stability-optimized) weights.

In [ ]:
# Load a structure
gfp_structure = get_gfp_structure()

# Prepare input
struct_input = InverseFoldingStructureInput(
    structure=gfp_structure,
    chain_ids=["A"],
)
inputs = InverseFoldingInput(inputs=[struct_input])

# Configure sampling with ProteinDPO weights (default)
config = ESMIF1SampleConfig(
    num_sequences_per_structure=3,
    temperature=0.1,
    weights_variant="protein_dpo",  # or "esmif" for vanilla ESM-IF1
    seed=42,
)

In [ ]:
# Run sampling
result = run_esm_if1_sample(inputs, config)

# Display designed sequences
for i, seq_res in enumerate(result.designed_sequences):
    print(f"\nStructure {i} designs:")
    for j, seq in enumerate(seq_res.sequences, 1):
        ll = seq_res.log_likelihoods[j - 1]
        display_seq = f"{seq[:50]}..." if len(seq) > 50 else seq
        print(f"  Design {j}: {display_seq}")
        print(f"            Length: {len(seq)} residues")
        print(f"            Log-likelihood: {ll:.4f}")

## 2. Score Sequences

Evaluate how well a sequence fits a given backbone structure. Uses the full complex structural context for scoring.

In [ ]:
# Score the first designed sequence against the structure
designed_seq = result.designed_sequences[0].sequences[0]

scoring_input = ESMIF1ScoringInput(
    sequence_structure_pairs=[
        SequenceStructurePair(
            sequence=designed_seq,
            structure=gfp_structure,
        )
    ]
)

scoring_config = ESMIF1ScoringConfig(
    weights_variant="protein_dpo",
)

score_result = run_esm_if1_score(scoring_input, scoring_config)

print(f"Avg log-likelihood: {score_result.scores[0].avg_log_likelihood:.4f}")
print(f"Perplexity: {score_result.scores[0].perplexity:.4f}")

In [ ]:
scoring_config_norm = ESMIF1ScoringConfig(
    weights_variant="protein_dpo",
    normalize=True,
)

score_result_norm = run_esm_if1_score(scoring_input, scoring_config_norm)

metrics = score_result_norm.scores[0].metrics
print(f"Avg log-likelihood: {metrics['avg_log_likelihood']:.4f}")
print(f"Native avg log-likelihood: {metrics['native_avg_log_likelihood']:.4f}")
print(f"Normalized score (variant - native): {metrics['normalized_score']:.4f}")

In [ ]:
# Export sampled sequences to FASTA
result.export("esm_if1_designs", export_path="./example_output", file_format="fasta")
print("Sequences exported to ./example_output/esm_if1_designs/")